In [ ]:
import sys
sys.path.insert(0,'/var/user-packages/usr/lib/python3.10/site-packages/')

In [ ]:
# POST http://169.254.111.111:22000/settings/common 
#{"ratio" : 24}

In [ ]:
# for fridge 
from dynio import *
import time
class Fridge:
    
    open_angle = 105
    close_angle = 10
    middle_angle = 50
    angle_tolerance = 0.5
    low_angle_speed = 20
    high_angle_speed = 50
    control_period = 0.01
    def __init__(self,serial_path="/dev/ttyACM0",motor_id=1):
      self.dxl_io = dxl.DynamixelIO(serial_path)
      self.mx_64 = self.dxl_io.new_mx64(motor_id, 1)  
    def _set_angle(self,target_angle):
      while 1:
        angle = self.mx_64.get_angle()
        if abs(angle - target_angle) < self.angle_tolerance:
          break
        if angle < self.middle_angle:
          self.mx_64.set_velocity(self.high_angle_speed)
        else:
          self.mx_64.set_velocity(self.low_angle_speed)
        self.mx_64.set_angle(target_angle)
        self.mx_64.write_control_table("P_Gain", 32)
        self.mx_64.write_control_table("D_Gain", 20)
        self.mx_64.torque_enable()
        time.sleep(self.control_period)
    def open_door(self):
      self._set_angle(self.open_angle)
    def close_door(self):
      self._set_angle(self.close_angle)
    def release_door(self):
      self.mx_64.write_control_table("P_Gain", 0)
      self.mx_64.write_control_table("D_Gain", 0)
      self.mx_64.set_angle(self.mx_64.get_angle())
      self.mx_64.torque_disable()
    def is_door_open(self):
        angle = self.mx_64.get_angle()
        if abs(self.open_angle - angle) < 2*self.angle_tolerance:
            return True
    def is_door_close(self):
        angle = self.mx_64.get_angle()
        if abs(self.close_angle - angle) < 2*self.angle_tolerance:
            return True

In [ ]:
import threading
from dobot_api import DobotApiDashboard, DobotApi, DobotApiMove, MyType,alarmAlarmJsonFile
from types import SimpleNamespace
import requests
import time
import tempfile
import urllib
from smb.SMBHandler import SMBHandler

class Dobot:
    slide_length = 800 
    ip = "169.254.111.111" # configured IP for LAN2 
    ip_debug_lan1 = "192.168.1.6" # debug ip fixed on LAN 1
    dashboardPort = 29999
    movePort = 30003
    feedPort = 30004
    web_port = 22000
    rail_offset = 0
    rail_acc = 1
    rail_speed = 1
    def change_lan2_ip(self, new_ip):
        payload = {    
        "dhcp": False,
        "gateway": new_ip,
        "ip": new_ip,
        "netmask": "255.255.0.0"
        }
        res = requests.post(
            f"http://{self.ip_debug_lan1}:{self.web_port}/interface/ethernet",
            json=payload)
        self.ip = new_ip
    
    def connect(self):
        self.dashboard = DobotApiDashboard(self.ip, self.dashboardPort)
        self.move = DobotApiMove(self.ip, self.movePort)
        self.feed = DobotApi(self.ip, self.feedPort)
        self.enable_rail(True)
    def get_pose(self):
        while True:
            res = self.dashboard.GetPose()
            if res.split(',')[-1] =='GetPose();':
                return [float(i) for i in res.replace('{','').replace('}','').split(',')[1:7]]
    def open_gripper(self): 
        self.dashboard.DO(1,0)
        self.dashboard.DO(2,0)
        time.sleep(0.3)
    def close_gripper(self):
        self.dashboard.DO(1,1)
        self.dashboard.DO(2,0)
        time.sleep(0.3)
    def get_robot_status(self):
        modesDict={
            "1" : ("ROBOT_MODE_INIT", "initialization"),
            "2" :	("ROBOT_MODE_BRAKE_OPEN","The brake release"),
            "3" :	("","Reserved"),
            "4" :	("ROBOT_MODE_DISABLED","Disabled (brake is not released)"),
            "5" :	("ROBOT_MODE_ENABLE","Enable (idle)"),
            "6" :	("ROBOT_MODE_BACKDRIVE","Dragging statej"),
            "7" :	("ROBOT_MODE_RUNNING","Running state"),
            "8" :	("ROBOT_MODE_RECORDING","Dragging recording"),
            "9" :	("ROBOT_MODE_ERROR","Alarm state"),
            "10" : ("ROBOT_MODE_PAUSE","pause state"),
            "11" : ("ROBOT_MODE_JOG","jogging state")
        }
        return modesDict[self.dashboard.RobotMode().split('{')[1].split('}')[0]]
    def _post(self,path,json_data=None):
        res = requests.post(
            f'http://{self.ip}:{self.web_port}/{path}',
            json=json_data)
        assert(res.status_code == 200)
        return res.json()
    def _get(self, path):
        res = requests.get(f'http://{self.ip}:{self.web_port}/{path}')
        assert(res.status_code == 200)
        return res.json()
    def set_global_speed_ratio(self, ratio=100):
        
        self.dashboard.SpeedFactor(ratio)

        # need to write a file on DOBOT using SMB protocol
        target_smb_url = f'smb://MG_400,{self.ip}/project/settings/common.json'
        data = {"ratio": ratio}
        tmp = tempfile.NamedTemporaryFile()
        with open(tmp.name, 'w') as f:
            f.write(f"""{{
    "ratio": {ratio}
}}
""")
        opener = urllib.request.build_opener(SMBHandler)

        with open(tmp.name, 'rb') as f:
            with opener.open(target_smb_url, data=f) as fh:
                print("write ratio to DOBOT")
        return self._post("settings/common", data)
    
    def get_emergency_stop_state(self):
        return self._get("panel/emergencyStop")
    def get_load_params(self):
        return self._get("settings/function/loadParams")
    def set_load_params(self, data):
        """ data = {
    "inertiaX": 0.10000000,
    "inertiaY": 0.10000000,
    "inertiaZ": 0,
    "loadValue": 0.2
}"""
        # need to write a file on DOBOT using SMB protocol
        target_smb_url = f'smb://MG_400,{self.ip}/project/settings/function/loadParams.json'
        tmp = tempfile.NamedTemporaryFile()
        with open(tmp.name, 'w') as f:
            f.write(f"""{{
    "inertiaX": {data["inertiaX"]},
    "inertiaY": {data["inertiaY"]},
    "inertiaZ": {data["inertiaZ"]},
    "loadValue": {data["loadValue"]}
}}""")
        opener = urllib.request.build_opener(SMBHandler)

        with open(tmp.name, 'rb') as f:
            with opener.open(target_smb_url, data=f) as fh:
                print("write data to DOBOT")
        return self._post("settings/common", data)
        
        return self._post("settings/function/loadParams",data)
    def get_user_param(self):
        return self._get("settings/driver/userParam")
    def get_connexion_state(self):
        return self._get("connection/state")
    def set_control_mode(self, data):
        """ data = {
    "controlMode": "enable"
    }
"""
        return self._post("settings/controlMode",data)
    

    def print_error_message(self):
        import json
        from files.alarm_controller import alarm_controller_list
        from files.alarm_servo import alarm_servo_list
        def convert_dict(alarm_list):
            alarm_dict = {}
            for i in alarm_list:
                alarm_dict[i["id"]] = i
            return alarm_dict
        def get_error(index, alarm_dict: dict, type_text):
            resp =  f"""ID:{index}
            Type:{type_text}"""
            if index in alarm_dict :
                resp += f"""
                Level:{alarm_dict[index]['level']}
                Solution:{alarm_dict[index]['en']['solution']}
            """
            else:
                resp+= f"""
                Level: Null
                Solution: Null
                """
                return resp
        alarm_controller_dict = convert_dict(alarm_controller_list)
        alarm_servo_dict = convert_dict(alarm_servo_list)
        error_list = json.loads(self.dashboard.GetErrorID().split("{")[1].split("}")[0])
        if error_list[0]:
            for i in error_list[0]:
                print(get_error(i,alarm_controller_dict, "Controller Error"))
        for m in range(1, len(error_list)):
            if error_list[m]:
                for n in range(len(error_list[m])):
                    print(get_error(n, alarm_servo_dict, "Servo Error"))

    def enable_rail(self,value=True):
        return self._post(
            "process/auxJoint/switch",
            {'auxJoint': value, 'addPoints': False}
            )
    def rail(self,value):
        return self.slide_length - value
        
    def calibrate_rail(self,true_pos=800):
        res = requests.post(
            f'http://{self.ip}:{self.web_port}/calibrate/homeAuxJoint')
        self._post(
            "calibrate/homeAuxJoint",
            None
            )
        data = self._get("protocol/exchange")
        actual_pos =self.rail(data["auxJoint"][0])
        self.rail_offset = true_pos - actual_pos
        
    def mov_arm_joints_and_rail(self,j1,j2,j3,j4,j_ext):
        rail_j = j_ext - self.rail_offset
        self.move.JointMovJ(j1,j2,j3,j4)
        self.move.MovJExt(self.rail(rail_j),self.rail_speed,self.rail_acc,1) # SpeedE=5 /100, AccE=5 / 100, SYNC=1 

    def mov_arm_coords_and_rail(self,x,y,z,r,y_ext,sync=True):
        rail_y = y_ext - self.rail_offset
        self.move.MovJ(x,y,z,r,) # {CP=1 [0:100], SpeedJ=50 [0:100], AccJ=20 [0:100]}
        self.move.MovJExt(self.rail(rail_y),self.rail_speed,self.rail_acc,1) # SpeedE=5 [0:100], AccE=5 [0:100], SYNC=1 [0,1]
        if sync:
            while not self.is_close_to(x,y,z,r,y_ext) :
                time.sleep(0.1)
        
    def is_close_to(self,x,y,z,r,y_ext):
        current_x,current_y,current_z,current_r,current_y_ext = self.get_arm_and_rail_coordinates()
        if abs(current_x-x)+abs(current_y-y)+abs(current_z-z)+abs(current_y_ext-y_ext)<2.0: 
            return True
        else:
            return False
    def get_arm_and_rail_joint_values(self):
        data= self._get("protocol/exchange")
        rail_j = self.rail(data["auxJoint"][0])
        j_ext = rail_j + self.rail_offset
        return data["jointCoordinate"]+[j_ext]
    
    def get_arm_and_rail_coordinates(self):
        data= self._get("protocol/exchange")
        x,y,z,r = data["cartesianCoordinate"]
        rail_y = self.rail(data["auxJoint"][0])
        y_ext = rail_y + self.rail_offset
        return [x,y,z,r,y_ext]
    
    def enable_robot(self):
        print(self.dashboard.GetErrorID())
        self.set_load_params(data = {
            "inertiaX": 0.1,
            "inertiaY": 0.1,
            "inertiaZ": 0,
            "loadValue": 0.1
        })

        #self.enable_rail(True)
        #end effector data
        end_effector = SimpleNamespace(
            load= 0.15,# in kg
            center_mass= SimpleNamespace(
                x= 0.1, # in mm ?
                y= 0.1,
                z= 0.1
            )
        )
        # self.dashboard.AccJ(50) # in %
        # self.dashboard.AccL(50)
        # self.dashboard.SpeedFactor(50)
        time.sleep(1)
        print(f'load={self.get_load_params()}')
        print(f'error={self.dashboard.GetErrorID()}')
        print(f'coord={self.get_arm_and_rail_coordinates()}')
        print(f'user={self.get_user_param()}')
        self.dashboard.SetPayload(end_effector.load,100)
        time.sleep(1)
        print(f'load={self.get_load_params()}')
        print(f'error={self.dashboard.GetErrorID()}')
        print(f'coord={self.get_arm_and_rail_coordinates()}')
        print(f'user={self.get_user_param()}')
        self.set_control_mode(data = {
        "controlMode": "enable"
        })
        time.sleep(1)
        print(f'load={self.get_load_params()}')
        print(f'error={self.dashboard.GetErrorID()}')
        print(f'coord={self.get_arm_and_rail_coordinates()}')
        print(f'user={self.get_user_param()}')
        self.dashboard.EnableRobot(end_effector.load, *end_effector.center_mass.__dict__.values())
        time.sleep(1)
        print(f'load={self.get_load_params()}')
        print(f'error={self.dashboard.GetErrorID()}')
        print(f'coord={self.get_arm_and_rail_coordinates()}')
        print(f'user={self.get_user_param()}')
        # #self.enable_rail(True)
        # self.dashboard.AccJ(50) # in %
        # self.dashboard.AccL(50)
        # self.dashboard.SpeedFactor(50)
        self.dashboard.SetPayload(end_effector.load,100)
        # time.sleep(1)
    def disable_robot(self) :
        self.set_control_mode(data = {
        "controlMode": "disable"
        })
        # #self.enable_rail(False)
        # #time.sleep(1)
        self.dashboard.DisableRobot()

In [ ]:
import math
import json
class Workspace:
    
    def __init__(self,fridge, arm) -> None:
        self.arm = arm
        self.fridge =fridge
        
        self.min_safe_total_y = 433
        self.max_safe_slide = 733+20
        self.slide_fridge = 360  # modified in calibration file to 430
        self.min_safe_slide = 0
        self.min_safe_x = -110
        self.min_safe_z = -120
        self.safe_plate = self.min_safe_x, -220, 0
        self.ot2_shift_slot_y = 132.49 # from CAD OT-2 file
        self.ot2_angular_offset = 1.15 # 1.1
        self.r_0 = -83.5
        
        self.plate_ot2_slot3 = 293, -132.5, -59.5
        self.plate_front_fridge_1_2 = 110, -170, 50,
        self.up_z_offset = 20
        self.cover_z_offset = 7

        self.plate_fridge_slot1 = 277, -80.3, 36.5   #277.184814453125, -80.25714874267578, 36.48689270019531

        self.fridge_angular_offset_1_2 = 1.5 

        self.plate_fridge_slot2 = 274, -228.3, 35.
        
        self.plate_front_fridge_3_4 = 110, -170, 0,
        
        self.fridge_angular_offset_3_4 = 1.5 

        self.plate_fridge_slot3 = 275, -83.0, -43.5

        self.plate_fridge_slot4 = 272.5, -231, -46.5

        self.pos_final_stock_cover_0= [94,
        260.43,
        -171.00,
        99.15,
        703.00]

        self.pos_final_stock_slot_0= [94,
        260.43,
        -163,
        99.15,
        703.00]
        

        self.pos_start_stock_cover= [140.0,
        132.43,
        0,
        53.2,
        703.00]
        
        self.temp_module_z_offset = 80
        self.temp_module_y_offset = -2
        self.temp_module_x_offset = 0
        self.compute_pos()

    def save_data(self, filename):
        self.compute_pos()
        data = { k:v for k,v in vars(self).items() if k not in ['arm','fridge'] }
        with open(filename,'w') as f:
            json.dump(data,f)
    
    def load_data(self, filename) :
        with open(filename,'r') as f:
            data = json.load(f)
        for k,v in data.items():
            self.__dict__[k] = v
        self.compute_pos()

    def compute_pos(self):

        self.r_90 = self.r_0 + 90

        self.pos_front_fridge_1_2 =  [
                self.plate_front_fridge_1_2[0],
                self.plate_front_fridge_1_2[1],
                self.plate_front_fridge_1_2[2],
                self.r_90,
                self.slide_fridge]
        
        self.pos_front_fridge_3_4 =  [
                self.plate_front_fridge_3_4 [0],
                self.plate_front_fridge_3_4 [1],
                self.plate_front_fridge_3_4 [2],
                self.r_90,
                self.slide_fridge]
    
        self.r_slot3 = self.ot2_angular_offset + self.r_90

        self.plate_ot2_slot2 = [
            self.plate_ot2_slot3[0] + self.ot2_shift_slot_y*math.sin(math.radians(self.ot2_angular_offset)),
            self.plate_ot2_slot3[1] + self.ot2_shift_slot_y*math.cos(math.radians(self.ot2_angular_offset)), 
            self.plate_ot2_slot3[2] + 1
        ]
        self.plate_ot2_slot1 =  [
            self.plate_ot2_slot3[0] + 2* self.ot2_shift_slot_y*math.sin(math.radians(self.ot2_angular_offset)),
            self.plate_ot2_slot3[1] + 2* self.ot2_shift_slot_y*math.cos(math.radians(self.ot2_angular_offset)), 
            self.plate_ot2_slot3[2] + 1.0
        ]
        

        self.r_fridge_slot1 = self.fridge_angular_offset_1_2 + self.r_90

        def compute_slot(x,y,z,r,slide):
            up = [x,y,z+self.up_z_offset,r,slide]
            final = [x,y,z,r,slide]
            cover = [x,y,z+self.cover_z_offset,r,slide]
            return up,final,cover
        
        self.pos_up_fridge_slot1, self.pos_final_fridge_slot1, self.pos_cover_fridge_slot1 = compute_slot(
            self.plate_fridge_slot1[0],
            self.plate_fridge_slot1[1],
            self.plate_fridge_slot1[2],
            self.r_fridge_slot1,
            self.pos_front_fridge_1_2[4]
        )

        self.r_fridge_slot2 = self.fridge_angular_offset_1_2 + self.r_90 
        
        self.pos_up_fridge_slot2, self.pos_final_fridge_slot2, self.pos_cover_fridge_slot2 = compute_slot(
            self.plate_fridge_slot2[0],
            self.plate_fridge_slot2[1],
            self.plate_fridge_slot2[2],
            self.r_fridge_slot2,
            self.pos_front_fridge_1_2[4]
        )

        self.r_fridge_slot3 = self.fridge_angular_offset_3_4 + self.r_90 

        self.pos_up_fridge_slot3, self.pos_final_fridge_slot3, self.pos_cover_fridge_slot3 = compute_slot(
            self.plate_fridge_slot3[0],
            self.plate_fridge_slot3[1],
            self.plate_fridge_slot3[2],
            self.r_fridge_slot3,
            self.pos_front_fridge_3_4[4]
        )
        
        self.r_fridge_slot4 = self.fridge_angular_offset_3_4 + self.r_90  

        self.pos_up_fridge_slot4, self.pos_final_fridge_slot4, self.pos_cover_fridge_slot4 = compute_slot(
            self.plate_fridge_slot4[0],
            self.plate_fridge_slot4[1],
            self.plate_fridge_slot4[2],
            self.r_fridge_slot4,
            self.pos_front_fridge_3_4[4]
        )
                

        self.pos_security = [ 
            self.safe_plate[0],
            self.safe_plate[1],
            self.safe_plate[2],
            self.r_0,
            self.max_safe_slide]

        self.pos_rail_origin = self.pos_security[0:-1] +[0]


        self.pos_start_OT2_slot3 = [
            140.0,
            self.plate_ot2_slot3[1],
                0,
            self.r_slot3,
            self.max_safe_slide]

        self.pos_start_OT2_slot2 = [
            220,
        self.plate_ot2_slot2[1],
                0,
            self.r_slot3,
            self.max_safe_slide]

        self.pos_start_OT2_slot1 = [
            140.0,
            self.plate_ot2_slot1[1],
                0,
            self.r_slot3,
            self.max_safe_slide]
        
        self.pos_start_OT2_temp_module_slot3 = [
            self.pos_start_OT2_slot3[0],
            self.pos_start_OT2_slot3[1],
            self.pos_start_OT2_slot3[2] + 25,
            self.pos_start_OT2_slot3[3],
            self.pos_start_OT2_slot3[4]]
        
        self.pos_up_OT2_slot1, self.pos_final_OT2_slot1, self.pos_cover_OT2_slot1 = compute_slot(
            self.plate_ot2_slot1[0],
            self.plate_ot2_slot1[1],
            self.plate_ot2_slot1[2],
            self.r_slot3,
            self.max_safe_slide
        )
        
        self.pos_up_OT2_temp_module_slot3, self.pos_final_OT2_temp_module_slot3, self.pos_cover_OT2_temp_module_slot3 = compute_slot(
            self.plate_ot2_slot3[0] + self.temp_module_x_offset,
            self.plate_ot2_slot3[1] + self.temp_module_y_offset,
            self.plate_ot2_slot3[2] + self.temp_module_z_offset,
            self.r_slot3,
            self.max_safe_slide
        )
        
        self.pos_up_OT2_slot2, self.pos_final_OT2_slot2, self.pos_cover_OT2_slot2 = compute_slot(
            self.plate_ot2_slot2[0],
            self.plate_ot2_slot2[1],
            self.plate_ot2_slot2[2],
            self.r_slot3,
            self.max_safe_slide
        )

        self.pos_up_OT2_slot3, self.pos_final_OT2_slot3, self.pos_cover_OT2_slot3 = compute_slot(
            self.plate_ot2_slot3[0],
            self.plate_ot2_slot3[1],
            self.plate_ot2_slot3[2],
            self.r_slot3,
            self.max_safe_slide
        )
        
        self.pos_up_stock_slot_0 = list(self.pos_final_stock_slot_0)
        self.pos_up_stock_slot_0[2] = self.pos_final_stock_slot_0[2]+20
        
    def calibrate(self,slot):
        x,y,z,r,slide = self.arm.get_arm_and_rail_coordinates()
        if slot == "final_stock_cover_0":
            self.pos_final_stock_cover_0 = [x,y,z,r,slide]
            self.compute_pos()
            return
        if slot == "start_stock_cover":
            self.pos_start_stock_cover = [x,y,z,r,slide]
            self.compute_pos()
            return
        if slot == "final_stock_slot_0":
            self.pos_final_stock_slot_0 = [x,y,z,r,slide]
            self.compute_pos()
            return
        if slot == "security":
            self.safe_plate = x,y,z
            self.r_0 = r
            self.max_safe_slide = slide
            self.compute_pos()
            return
        if slot == "final_OT2_slot3":
            self.plate_ot2_slot3 = x,y,z
            self.max_safe_slide = slide
            self.ot2_angular_offset = r - self.r_90
            self.compute_pos()
            return
        if slot == "final_OT2_slot1":
            self.ot2_angular_offset =math.degrees(math.atan2(y - self.plate_ot2_slot3[1],x - self.plate_ot2_slot3[0]))
            self.compute_pos()
            return
        if slot == "final_fridge_slot1":
            self.plate_fridge_slot1 = x,y,z
            self.fridge_angular_offset_1_2 = r - self.r_90
            self.compute_pos()
            return
        if slot == "final_fridge_slot2":
            self.plate_fridge_slot2 = x,y,z
            self.r_fridge_slot2 = r
            self.compute_pos()
            return
        if slot == "final_fridge_slot3":
            self.plate_fridge_slot3 = x,y,z
            self.fridge_angular_offset_3_4 = r - self.r_90
            self.r_fridge_slot3 = r
            self.compute_pos()
            return
        if slot == "final_fridge_slot4":
            self.plate_fridge_slot4 = x,y,z
            self.r_fridge_slot4 = r
            self.compute_pos()
            return
        print("unknown position name")

        


    
    def safe(self, action):
        if action == self.fridge.open_door or action == self.fridge.close_door :
            x,y,z,r,y_slide = self.arm.get_arm_and_rail_coordinates()
            if y + y_slide > self.min_safe_total_y:
                action()
            else:
                print("robot not safe for Fridge action ")
        else:
            print("unknown action")
    
    def do_action_slot(self, slot="slot1", griper_action=None):
        
        def final_move(pos_start,pos_up,pos_final,griper_action=None):
            self.arm.mov_arm_coords_and_rail(*pos_start)
            self.arm.mov_arm_coords_and_rail(*pos_up)
            self.arm.mov_arm_coords_and_rail(*pos_final)
            if griper_action == "break":
                return
            if griper_action :
                griper_action()
            self.arm.mov_arm_coords_and_rail(*pos_up)
            self.arm.mov_arm_coords_and_rail(*pos_start)
        
        def go_back(pos_start,pos_up,pos_final):
            if self.arm.is_close_to(*pos_final):
                self.arm.mov_arm_coords_and_rail(*pos_up)
                self.arm.mov_arm_coords_and_rail(*pos_start)

        def go_back_slot_and_cover(pos_slot_start, pos_up, pos_final_cover, pos_final_slot):
            for pos_final in [pos_final_cover, pos_final_slot]:
                go_back(pos_slot_start,
                        pos_up,
                        pos_final)
        
        if slot == "security" :
            if griper_action :
                griper_action()
            go_back_slot_and_cover(
                self.pos_start_stock_cover,
                self.pos_up_stock_slot_0,
                self.pos_final_stock_cover_0,
                self.pos_final_stock_slot_0
            )
            
            go_back_slot_and_cover(
                self.pos_start_OT2_slot1,
                self.pos_up_OT2_slot1,
                self.pos_cover_OT2_slot1,
                self.pos_final_OT2_slot1
            )
            go_back_slot_and_cover(
                self.pos_start_OT2_slot2,
                self.pos_up_OT2_slot2,
                self.pos_cover_OT2_slot2,
                self.pos_final_OT2_slot2
            )
            go_back_slot_and_cover(
                self.pos_start_OT2_slot3,
                self.pos_up_OT2_slot3,
                self.pos_cover_OT2_slot3,
                self.pos_final_OT2_slot3
            )
            go_back_slot_and_cover(
                    self.pos_start_OT2_temp_module_slot3,
                    self.pos_up_OT2_temp_module_slot3,
                    self.pos_cover_OT2_temp_module_slot3,
                    self.pos_final_OT2_temp_module_slot3
            )
            go_back_slot_and_cover(
                    self.pos_front_fridge_1_2,
                    self.pos_up_fridge_slot1,
                    self.pos_cover_fridge_slot1,
                    self.pos_final_fridge_slot1
            )
            go_back_slot_and_cover(
                    self.pos_front_fridge_1_2,
                    self.pos_up_fridge_slot2,
                    self.pos_cover_fridge_slot2,
                    self.pos_final_fridge_slot2
            )
            go_back_slot_and_cover(
                    self.pos_front_fridge_3_4,
                    self.pos_up_fridge_slot3,
                    self.pos_cover_fridge_slot3,
                    self.pos_final_fridge_slot3
            )
            go_back_slot_and_cover(
                    self.pos_front_fridge_3_4,
                    self.pos_up_fridge_slot4,
                    self.pos_cover_fridge_slot4,
                    self.pos_final_fridge_slot4
            )
            
            if self.arm.is_close_to(*self.pos_start_OT2_temp_module_slot3) :
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot3)
            
            if self.arm.is_close_to(*self.pos_start_stock_cover) :
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot1)
            if self.arm.is_close_to(*self.pos_start_OT2_slot1) :
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot3)
            if self.arm.is_close_to(*self.pos_start_OT2_slot2) :
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot3)
            if self.arm.is_close_to(*self.pos_start_OT2_slot3) :
                self.arm.mov_arm_coords_and_rail(*self.pos_security)
            if self.arm.is_close_to(*self.pos_front_fridge_1_2) :
                self.arm.mov_arm_coords_and_rail(*self.pos_security)
                self.safe(self.fridge.close_door)
            if self.arm.is_close_to(*self.pos_front_fridge_3_4) :
                self.arm.mov_arm_coords_and_rail(*self.pos_security)
                self.safe(self.fridge.close_door)
        
        elif slot == "stock_slot_0":
            pos_final = self.pos_final_stock_slot_0
            if self.arm.is_close_to(*self.pos_security):
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot3)
            if  ( self.arm.is_close_to(*self.pos_start_OT2_slot2) or 
                    self.arm.is_close_to(*self.pos_start_OT2_slot1) or 
                    self.arm.is_close_to(*self.pos_start_OT2_slot3) ):
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot1)
            if self.arm.is_close_to(*self.pos_start_OT2_slot1) :
                self.arm.mov_arm_coords_and_rail(*self.pos_start_stock_cover)
            if self.arm.is_close_to(*self.pos_start_stock_cover):
                final_move(
                    self.pos_start_stock_cover,
                    self.pos_up_stock_slot_0,
                    pos_final,
                    griper_action
                )
                if griper_action == "break":
                    return
                self.arm.mov_arm_coords_and_rail(*self.pos_start_stock_cover)
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot1)
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot3)
                self.arm.mov_arm_coords_and_rail(*self.pos_security)

        elif slot == "slot1" or slot == "cover_slot1":
            if slot == "slot1":
                pos_final = self.pos_final_OT2_slot1
            else:
                pos_final = self.pos_cover_OT2_slot1
            if self.arm.is_close_to(*self.pos_security):
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot3)
            if  ( self.arm.is_close_to(*self.pos_start_OT2_slot2) or 
                    self.arm.is_close_to(*self.pos_start_OT2_slot1) or 
                    self.arm.is_close_to(*self.pos_start_OT2_slot3) ):
                final_move(
                    self.pos_start_OT2_slot1,
                    self.pos_up_OT2_slot1,
                    pos_final,
                    griper_action
                )
                if griper_action == "break":
                    return
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot3)
                self.arm.mov_arm_coords_and_rail(*self.pos_security)
        elif slot == "slot2" or slot == "cover_slot2":
            if slot == "slot2":
                pos_final = self.pos_final_OT2_slot2
            else:
                pos_final = self.pos_cover_OT2_slot2
            if self.arm.is_close_to(*self.pos_security):
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot3)
            if ( self.arm.is_close_to(*self.pos_start_OT2_slot3) or 
            self.arm.is_close_to(*self.pos_start_OT2_slot1) or 
            self.arm.is_close_to(*self.pos_start_OT2_slot2) ) :
                final_move(
                    self.pos_start_OT2_slot2,
                    self.pos_up_OT2_slot2,
                    pos_final,
                    griper_action
                    )
                if griper_action == "break":
                    return
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot3)
            self.arm.mov_arm_coords_and_rail(*self.pos_security)
        elif slot == "slot3" or slot == "cover_slot3":
            if slot == "slot3":
                pos_final = self.pos_final_OT2_slot3
            else:
                pos_final = self.pos_cover_OT2_slot3
            if ( self.arm.is_close_to(*self.pos_security) or 
            self.arm.is_close_to(*self.pos_start_OT2_slot2) or
            self.arm.is_close_to(*self.pos_start_OT2_slot3) ):
                final_move(
                    self.pos_start_OT2_slot3,
                    self.pos_up_OT2_slot3,
                    pos_final,
                    griper_action
                    )
                if griper_action == "break":
                    return
            self.arm.mov_arm_coords_and_rail(*self.pos_security)
        elif slot == "temp_module_slot3" or slot == "cover_temp_module_slot3":
            if slot == "temp_module_slot3":
                pos_final = self.pos_final_OT2_temp_module_slot3
            else:
                pos_final = self.pos_cover_OT2_temp_module_slot3
            if self.arm.is_close_to(*self.pos_security):
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot3)
            if  ( self.arm.is_close_to(*self.pos_start_OT2_slot2) or 
                    self.arm.is_close_to(*self.pos_start_OT2_slot1) or 
                    self.arm.is_close_to(*self.pos_start_OT2_temp_module_slot3) or 
                    self.arm.is_close_to(*self.pos_start_OT2_slot3) ):
                final_move(
                    self.pos_start_OT2_temp_module_slot3,
                    self.pos_up_OT2_temp_module_slot3,
                    pos_final,
                    griper_action
                )
                if griper_action == "break":
                    return
                self.arm.mov_arm_coords_and_rail(*self.pos_start_OT2_slot3)
                self.arm.mov_arm_coords_and_rail(*self.pos_security)
        elif slot == "fridge_slot1" or slot == "cover_fridge_slot1":
            if slot == "fridge_slot1":
                pos_final = self.pos_final_fridge_slot1
            else:
                pos_final = self.pos_cover_fridge_slot1
            if self.arm.is_close_to(*self.pos_security) :
                if self.fridge.is_door_close():
                    self.safe(self.fridge.open_door)
            if self.fridge.is_door_open():
                final_move(
                    self.pos_front_fridge_1_2,
                    self.pos_up_fridge_slot1,
                    pos_final,
                    griper_action
                    )
                if griper_action == "break":
                    return
            self.arm.mov_arm_coords_and_rail(*self.pos_security)
            self.safe(self.fridge.close_door)
        elif slot == "fridge_slot2" or slot == "cover_fridge_slot2":
            if slot == "fridge_slot2":
                pos_final = self.pos_final_fridge_slot2
            else:
                pos_final = self.pos_cover_fridge_slot2
            if self.arm.is_close_to(*self.pos_security) :
                if self.fridge.is_door_close():
                    self.safe(self.fridge.open_door)
            if self.fridge.is_door_open():
                final_move(
                    self.pos_front_fridge_1_2,
                    self.pos_up_fridge_slot2,
                    pos_final,
                    griper_action
                )
                if griper_action == "break":
                    return
            self.arm.mov_arm_coords_and_rail(*self.pos_security)
            self.safe(self.fridge.close_door)
        elif slot == "fridge_slot3" or slot == "cover_fridge_slot3":
            if slot == "fridge_slot3":
                pos_final = self.pos_final_fridge_slot3
            else:
                pos_final = self.pos_cover_fridge_slot3
            if self.arm.is_close_to(*self.pos_security) :
                if self.fridge.is_door_close():
                    self.safe(self.fridge.open_door)
            if self.fridge.is_door_open():
                final_move(
                    self.pos_front_fridge_3_4,
                    self.pos_up_fridge_slot3,
                    pos_final,
                    griper_action
                    )
                if griper_action == "break":
                    return
            self.arm.mov_arm_coords_and_rail(*self.pos_security)
            self.safe(self.fridge.close_door)
        elif slot == "fridge_slot4" or slot == "cover_fridge_slot4":
            if slot == "fridge_slot4":
                pos_final = self.pos_final_fridge_slot4
            else:
                pos_final = self.pos_cover_fridge_slot4
            if self.arm.is_close_to(*self.pos_security) :
                if self.fridge.is_door_close():
                    self.safe(self.fridge.open_door)
            if self.fridge.is_door_open():
                final_move(
                    self.pos_front_fridge_3_4,
                    self.pos_up_fridge_slot4,
                    pos_final,
                    griper_action
                )
                if griper_action == "break":
                    return
            self.arm.mov_arm_coords_and_rail(*self.pos_security)
            self.safe(self.fridge.close_door)
        else :
            print(f"slot {slot} is not know")
            pass    

In [ ]:
fridge = Fridge(serial_path="/dev/ttyACM0")

In [ ]:
arm = Dobot()

In [ ]:
ws = Workspace(fridge, arm)

In [ ]:
ws.load_data('20250723_calibration.json')

In [ ]:
def step(axis=0, step=0.01):
    pos = arm.get_arm_and_rail_coordinates()
    print(pos)
    pos[axis] += step
    arm.mov_arm_coords_and_rail(*pos)

In [ ]:
arm.connect()
arm.get_robot_status()

In [ ]:
arm.get_load_params()

In [ ]:
arm.get_arm_and_rail_coordinates()

In [ ]:
ws.safe(fridge.close_door)

In [ ]:
arm.enable_robot()

In [ ]:
arm.get_robot_status()

In [ ]:
#slow down movements
arm.set_global_speed_ratio(100)
time.sleep(2)
arm.set_global_speed_ratio(22)

In [ ]:
arm.mov_arm_coords_and_rail(*ws.pos_security)

# CLEAR ERROR

In [ ]:
arm.dashboard.ClearError()
#arm.dashboard.ResetRobot()

In [ ]:
arm.dashboard.GetErrorID()

In [ ]:
arm.disable_robot()

In [ ]:
import ipywidgets as widgets

In [ ]:
output = widgets.Output()
button = widgets.Button(
    description='Click me',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Click me',
    icon='check', # (FontAwesome names without the `fa-` prefix),
)
button2 = widgets.Button(
    description='Click me 2',
    disabled=False,
    button_style='success', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Click me',
    icon='check' # (FontAwesome names without the `fa-` prefix)
)
display(button,button2,output)

def on_button_clicked(b):
    with output:
        print("Button clicked.")

button.on_click(on_button_clicked)

# GET ERROT ID

In [ ]:
arm.dashboard.GetErrorID()

In [ ]:
arm.print_error_message()

In [ ]:
arm.dashboard.ClearError()

In [ ]:
arm.get_arm_and_rail_coordinates()

In [ ]:
#arm.get_arm_and_rail_joint_values()

In [ ]:
#arm.mov_arm_joints_and_rail(-90,0,0,7,700)

In [ ]:
arm.dashboard.ResetRobot()

 # Step to calibrate position

In [ ]:
#arm.set_global_speed_ratio(22)
#arm.dashboard.AccJ(50) # in %
#arm.dashboard.AccL(50)
#arm.dashboard.SpeedFactor(50)
# Calibration à l'aide la commande step(x;y) avec y = l'axe (0 = Arm X; 1 = Arm Y; 2 = Arm Z; 3 = Gripper; 4 = Slider) et x la valeur en mm
time.sleep(2)
step(0,0.25)

In [ ]:
ws.pos_start_OT2_temp_module_slot3

In [ ]:
arm.get_arm_and_rail_coordinates()

# Save calibration 

In [ ]:
ws.pos_final_fridge_slot4

In [ ]:
ws.calibrate("final_fridge_slot4")

In [ ]:
arm.get_arm_and_rail_coordinates()

In [ ]:
ws.pos_final_fridge_slot4

In [ ]:
ws.compute_pos()

In [ ]:
ws.save_data('20250723_calibration.json')

# Security Pos

In [ ]:
arm.set_global_speed_ratio(100)
time.sleep(2)
arm.set_global_speed_ratio(22)

In [ ]:
arm.mov_arm_coords_and_rail(*ws.pos_security)

In [ ]:
arm.dashboard.SetPayload(0.2,100)

In [ ]:
if fridge.is_door_open():
    arm.set_global_speed_ratio(22)
    arm.mov_arm_coords_and_rail(*ws.pos_front_fridge_1_2)
else:
    print("Fridge Door is not open")

 # Rail Origin Pos

In [ ]:
arm.set_global_speed_ratio(22)

In [ ]:
ws.pos_rail_origin

In [ ]:
if fridge.is_door_close():
    arm.mov_arm_coords_and_rail(*ws.pos_rail_origin)
else:
    print("Fridge Door is not closed")

# SLOT 3

In [ ]:
arm.mov_arm_coords_and_rail(*ws.pos_security)

In [ ]:
arm.mov_arm_coords_and_rail(*ws.pos_start_OT2_slot3)

In [ ]:
arm.mov_arm_coords_and_rail(*ws.pos_up_OT2_slot3)

In [ ]:
arm.mov_arm_coords_and_rail(*ws.pos_final_OT2_slot3)

In [ ]:
arm.mov_arm_coords_and_rail(*ws.pos_cover_OT2_slot3)

In [ ]:
arm.close_gripper()

In [ ]:
arm.open_gripper()

# SLOT 1

In [ ]:
#arm.mov_arm_coords_and_rail(*pos_start_OT2_slot3)
#time.sleep(2)
#arm.mov_arm_coords_and_rail(*pos_start_OT2_slot2)
#time.sleep(2)
arm.mov_arm_coords_and_rail(*ws.pos_start_OT2_slot1)

In [ ]:
arm.mov_arm_coords_and_rail(*ws.pos_start_OT2_slot3)

In [ ]:
arm.mov_arm_coords_and_rail(*ws.pos_up_OT2_slot1)

In [ ]:
arm.mov_arm_coords_and_rail(*ws.pos_final_OT2_slot1)

# Cover stock slot 0

In [ ]:
time.sleep(2)
arm.mov_arm_coords_and_rail(*ws.pos_start_OT2_slot3)

In [ ]:
arm.mov_arm_coords_and_rail(*ws.pos_start_OT2_slot1)

In [ ]:
time.sleep(2)
arm.mov_arm_coords_and_rail(*ws.pos_start_stock_cover)

In [ ]:
time.sleep(2)
arm.mov_arm_coords_and_rail(*ws.pos_up_stock_slot_0)

In [ ]:
time.sleep(2)
arm.mov_arm_coords_and_rail(*ws.pos_final_stock_slot_0)

# Temp Module SLOT 3

In [ ]:
arm.mov_arm_coords_and_rail(*ws.pos_start_OT2_slot3)

In [ ]:
time.sleep(2)
arm.mov_arm_coords_and_rail(*ws.pos_start_OT2_temp_module_slot3)

In [ ]:
time.sleep(2)
arm.mov_arm_coords_and_rail(*ws.pos_up_OT2_temp_module_slot3)

In [ ]:
time.sleep(2)
arm.mov_arm_coords_and_rail(*ws.pos_final_OT2_temp_module_slot3)

In [ ]:
ws.temp_module_y_offset += 0.5     
ws.compute_pos()

In [ ]:
arm.close_gripper()

In [ ]:
arm.open_gripper()

In [ ]:
ws.pos_final_OT2_temp_module_slot3

In [ ]:
arm.get_arm_and_rail_coordinates()

In [ ]:
ws.pos_up_OT2_temp_module_slot3

# SLOT 2

In [ ]:
arm.mov_arm_coords_and_rail(*ws.pos_start_OT2_slot3)
time.sleep(2)
arm.mov_arm_coords_and_rail(*ws.pos_start_OT2_slot2)

In [ ]:
arm.mov_arm_coords_and_rail(*ws.pos_up_OT2_slot2)

In [ ]:
arm.mov_arm_coords_and_rail(*ws.pos_final_OT2_slot2)

# Fridge Slot 1

In [ ]:
ws.safe(fridge.open_door)

In [ ]:
ws.safe(fridge.close_door)

In [ ]:
if fridge.is_door_open():
    arm.mov_arm_coords_and_rail(*ws.pos_front_fridge_1_2)
else:
    print("Fridge Door is not open")

In [ ]:
if fridge.is_door_open():
    arm.mov_arm_coords_and_rail(*ws.pos_up_fridge_slot1)

In [ ]:
if fridge.is_door_open():
    arm.mov_arm_coords_and_rail(*ws.pos_final_fridge_slot1)

In [ ]:
arm.close_gripper()

In [ ]:
arm.open_gripper()

# Fridge Slot 2

In [ ]:
if fridge.is_door_open():
    arm.mov_arm_coords_and_rail(*ws.pos_front_fridge_1_2)
else:
    print("Fridge Door is not open")

In [ ]:
if fridge.is_door_open():
    arm.mov_arm_coords_and_rail(*ws.pos_up_fridge_slot2)

In [ ]:
if fridge.is_door_open():
    arm.mov_arm_coords_and_rail(*ws.pos_final_fridge_slot2)

In [ ]:
arm.close_gripper()

In [ ]:
arm.open_gripper()

# Fridge Slot 3

In [ ]:
ws.safe(fridge.open_door)

In [ ]:
if fridge.is_door_open():
    arm.mov_arm_coords_and_rail(*ws.pos_front_fridge_3_4)
else:
    print("Fridge Door is not open")

In [ ]:
if fridge.is_door_open():
    arm.mov_arm_coords_and_rail(*ws.pos_up_fridge_slot3)

In [ ]:
if fridge.is_door_open():
    arm.mov_arm_coords_and_rail(*ws.pos_final_fridge_slot3)

In [ ]:
arm.close_gripper()

In [ ]:
arm.open_gripper()

# Fridge Slot 4

In [ ]:
ws.safe(fridge.open_door)

In [ ]:
if fridge.is_door_open():
    arm.mov_arm_coords_and_rail(*ws.pos_front_fridge_3_4)
else:
    print("Fridge Door is not open")

In [ ]:
if fridge.is_door_open():
    arm.mov_arm_coords_and_rail(*ws.pos_up_fridge_slot4)

In [ ]:
ws.pos_up_fridge_slot4

In [ ]:
ws.pos_final_fridge_slot4

In [ ]:
if fridge.is_door_open():
    arm.mov_arm_coords_and_rail(*ws.pos_final_fridge_slot4)

In [ ]:
arm.close_gripper()

In [ ]:
arm.open_gripper()

# Gripper

In [ ]:
arm.close_gripper()

In [ ]:
arm.open_gripper()

In [ ]:
time.sleep(5)
arm.close_gripper()
time.sleep(5)
arm.open_gripper()

In [ ]:
def pause():
    time.sleep(10)

In [ ]:
def pause_griper_test():
    time.sleep(3)
    arm.close_gripper()
    time.sleep(3)
    arm.open_gripper()

In [ ]:
ws.load_data("test.json")

# Do action

In [ ]:
time.sleep(2)

ws.do_action_slot("temp_module_slot3",arm.open_gripper)

In [ ]:
time.sleep(2)

#ws.do_action_slot("fridge_slot1",pause_griper_test)
ws.do_action_slot("fridge_slot2",arm.close_gripper)
ws.do_action_slot("stock_slot_0",arm.open_gripper)
#ws.do_action_slot("slot1",arm.open_gripper)

In [ ]:
ws.do_action_slot("security",arm.close_gripper)

In [ ]:
ws.do_action_slot("stock_slot_0",arm.open_gripper)
#ws.do_action_slot("temp_module_slot3",arm.open_gripper)

In [ ]:
ws.do_action_slot("slot1",arm.open_gripper)

In [ ]:
ws.do_action_slot("fridge_slot1",arm.open_gripper)

In [ ]:
ws.do_action_slot("fridge_slot2",arm.close_gripper)

In [ ]:
ws.safe(fridge.open_door)

In [ ]:
# gripper setup
# check using dmesg for example the path of usb serial port assigned for usb-2-RS-485 converter

serial_path="/dev/ttyUSB0"

from pymodbus.client.sync import ModbusSerialClient as ModbusClient
modbus = ModbusClient(method='rtu', port=serial_path, baudrate=115200, timeout=10)
modbus.connect()

In [ ]:
# full init gripper
res =modbus.write_register(address=0x0100,unit=1,value=0xa5)
assert(hex(res.value)=="0xa5")

In [ ]:
# Turn On I/O mode
res =modbus.write_register(address=0x0402,unit=1,value=1)
res.value

In [ ]:
# save gripper settings
res =modbus.write_register(address=0x0300,unit=1,value=1)
res.value

In [ ]:
# read gripper reference position (range 0-1000 (‰))
res = modbus.read_holding_registers(address=0x0103,unit=1,count=1)
res.registers

In [ ]:
# read gripper current position (range 0-1000 (‰))
res = modbus.read_holding_registers(address=0x0202,unit=1,count=1)
res.registers

In [ ]:
# read limit force (%) to finger stop or object detection 
res = modbus.read_holding_registers(address=0x0101,unit=1,count=1)
res.registers

In [ ]:
# read gripper speed (%)
res = modbus.read_holding_registers(address=0x0104,unit=1,count=1)
res.registers

In [ ]:
modbus.close()